# Perron: Google Gemma 4 E4B Pointwise Scaling Control & Capacity Floor Benchmark
### Protocol Reference: PREREG-E4B-CONTROL-v1 (Held-Out SWE-bench Verified Cohort)

**Author**: Sultan Haikal (GitHub: `@yvliet`)  
**Target**: Track 2 (Gemma 4 Developer Agent Paper Track)  
**Hardware Target**: Dual NVIDIA Tesla T4 GPUs (2x 15GB GDDR6) or Single P100 / T4  
**License**: Apache-2.0

---

## 1. Executive Protocol & Theoretical Invariants

This notebook executes the **Action 2.1 Positive Scaling Control** for the Perron code localization monograph.

### The Core Scientific Question
In SWE-bench Verified evaluations, zero-shot pointwise reranking with `google/gemma-4-e2b-it` (2.0B PLE) tied with lexical BM25 (MRR 0.3409 vs. 0.3439, AUC 0.7796 vs. 0.8245), failing to convert non-lexical graph-expanded candidates (Stratum C) into top-ranked hits.

Adversarial peer review questioned whether this null result indicates that graph expansion is inherently flawed, or whether it reflects an **uninformative capacity floor** of 2B models.

Here, we scale model capacity by $2\times$ to **`google/gemma-4-e4b-it`** (4.0B PLE, 128K context, native `<|think|>` tokens) across candidate pools to test:
1. **$H_1$ (Capacity Ceiling Breakout)**: E4B demonstrates statistically significant candidate discrimination (AUC $> 0.80$) on Stratum C non-seed defect locations.
2. **$H_2$ (Graph-Directed Conversion)**: Top-5 defect localization accuracy under Arm A4 (Spectral Graph) exceeds Arm A2 (Depth-BM25) under E4B capacity.

In [ ]:
# Step 1: Environment Setup, CUDA Diagnostics & Hugging Face Authentication
import os
import sys
import time
import json
import re
import gc
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Authenticate with Hugging Face via Kaggle Secrets if available
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
        os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
        print('[+] Successfully attached HF_TOKEN from Kaggle Secrets.')
except Exception:
    print('[-] No Kaggle Secrets found. Proceeding with public/local models.')

print(f'Python Version : {sys.version.split()[0]}')
print(f'PyTorch Version: {torch.__version__}')
cuda_available = torch.cuda.is_available()
gpu_count = torch.cuda.device_count() if cuda_available else 0
print(f'CUDA Available : {cuda_available} (GPU Count: {gpu_count})')

if cuda_available:
    for i in range(gpu_count):
        p = torch.cuda.get_device_properties(i)
        print(f'  [GPU {i}]: {p.name} | Total VRAM: {p.total_memory / (1024**3):.2f} GB')
else:
    print('WARNING: CUDA is not available. GPU acceleration is strongly recommended for E4B.')


In [ ]:
# Step 2: Ensure Dependencies
!pip install -q accelerate bitsandbytes scipy transformers


In [ ]:
# Step 3: Dataset Ingestion & Evaluation Bundle Resolver
# Locates stratum_c_eval_bundle.json or gemma4_eval_batch_344.json across Kaggle mount points

candidate_paths = [
    Path('/kaggle/input/perron-gemma4-benchmark/stratum_c_eval_bundle.json'),
    Path('/kaggle/input/stratum_c_eval_bundle.json'),
    Path('data_release/stratum_c_eval_bundle.json'),
    Path('../data_release/stratum_c_eval_bundle.json'),
    Path('data/gemma4_eval_batch_344.json'),
    Path('../data/gemma4_eval_batch_344.json'),
]

bundle_path = None
for cp in candidate_paths:
    if cp.exists():
        bundle_path = cp
        break

if bundle_path is None:
    base_in = Path('/kaggle/input')
    if base_in.exists():
        for found in base_in.rglob('*eval_bundle*.json'):
            if found.is_file():
                bundle_path = found
                break

if bundle_path is None:
    raise FileNotFoundError(
        'Could not find evaluation bundle. Please add the dataset `perron-gemma4-benchmark` '
        'or upload `stratum_c_eval_bundle.json` to /kaggle/input.'
    )

print(f'[+] Ingesting evaluation bundle from: {bundle_path}')
with open(bundle_path, 'r', encoding='utf-8') as f:
    bundle_raw = json.load(f)

tasks = bundle_raw.get('tasks', bundle_raw) if isinstance(bundle_raw, dict) else bundle_raw
print(f'[+] Successfully loaded {len(tasks)} tasks.')
t0 = tasks[0]
print(f'    Sample task ID : {t0.get("instance_id")}')
print(f'    Candidate pools: {list(t0.get("pools", {}).keys()) if "pools" in t0 else "Custom batch format"}')


In [ ]:
# Step 4: Model Checkpoint Resolver & Model Loader for Google Gemma 4 E4B
from transformers import AutoModelForCausalLM, AutoTokenizer

TARGET_MODEL_ID = 'google/gemma-4-e4b-it'

def resolve_model_path(model_id: str) -> str:
    # 1. Local working directory check
    if Path(model_id).exists() and (Path(model_id) / 'config.json').exists():
        return model_id
    # 2. Kaggle dataset / model mount check
    base_in = Path('/kaggle/input')
    if base_in.exists():
        for p in base_in.rglob('config.json'):
            p_str = str(p.parent).lower()
            if 'e4b' in p_str and 'gemma' in p_str:
                print(f'[+] Resolved local Kaggle model mount: {p.parent}')
                return str(p.parent)
    # 3. Fallback to Hugging Face Hub
    return model_id

resolved_model_path = resolve_model_path(TARGET_MODEL_ID)
print(f'[+] Loading Gemma 4 E4B from: {resolved_model_path}')

tokenizer = AutoTokenizer.from_pretrained(resolved_model_path, padding_side='left')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Identify Yes / No token IDs for binary log-odds scoring
yes_token_ids, no_token_ids = set(), set()
for word in ['Yes', ' Yes', 'YES', ' yes']:
    enc = tokenizer.encode(word, add_special_tokens=False)
    if enc: yes_token_ids.add(enc[-1])
for word in ['No', ' No', 'NO', ' no']:
    enc = tokenizer.encode(word, add_special_tokens=False)
    if enc: no_token_ids.add(enc[-1])

yes_ids = sorted(list(yes_token_ids))
no_ids = sorted(list(no_token_ids))
print(f'[+] Resolved Token IDs -> Yes: {yes_ids}, No: {no_ids}')

# Load model in bfloat16 (or float16 fallback) with automatic multi-GPU dispatch
torch_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    resolved_model_path,
    torch_dtype=torch_dtype,
    device_map='auto',
    trust_remote_code=True,
)
model.eval()
print(f'[+] Model successfully loaded in {torch_dtype}. Device footprint:')
if hasattr(model, 'hf_device_map'):
    for mod, dev in list(model.hf_device_map.items())[:6]:
        print(f'    {mod} -> {dev}')


In [ ]:
# Step 5: Prompt Formatter & Batched Scoring Engine

PROMPT_TEMPLATE = (
    "You are an expert software engineer analyzing a bug report.\n\n"
    "Problem Statement:\n{problem_statement}\n\n"
    "Candidate Code Element:\n{candidate_breadcrumb}\n\n"
    "Does this candidate code element contain or directly cause the reported bug? Answer 'Yes' or 'No'."
)

def format_breadcrumb(cand: dict) -> str:
    if 'breadcrumb' in cand and cand['breadcrumb']:
        return cand['breadcrumb']
    ident = cand.get('identifier') or cand.get('symbol_name') or 'unknown'
    fpath = cand.get('file_path') or cand.get('file') or 'unknown'
    l_start = cand.get('line_start', 1)
    l_end = cand.get('line_end', l_start)
    snippet = cand.get('snippet') or cand.get('code') or f'# Symbol {ident} in {fpath}'
    return f'File: {fpath} (Lines {l_start}-{l_end})\nSymbol: {ident}\n```python\n{snippet}\n```'

def truncate_text(text: str, max_chars: int = 3000) -> str:
    if not text: return ''
    clean = re.sub(r'\s+', ' ', text).strip()
    if len(clean) <= max_chars: return clean
    half = (max_chars - 60) // 2
    return clean[:half] + ' ... [TRUNCATED] ... ' + clean[-half:]

@torch.inference_mode()
def score_prompt_batch(prompts: list[str]) -> list[tuple[float, float, int]]:
    """
    Executes batched forward pass with logits_to_keep=1 for bounded peak VRAM.
    Computes log-odds: s = logsumexp(z[Y]) - logsumexp(z[N]).
    """
    inputs = tokenizer(
        prompts,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=2048,
    )
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    try:
        outputs = model(**inputs, logits_to_keep=1)
        logits = outputs.logits[:, -1, :].to(torch.float32)
    except TypeError:
        outputs = model(**inputs)
        logits = outputs.logits[:, -1, :].to(torch.float32)

    probs = torch.softmax(logits, dim=-1)
    batch_results = []
    for i in range(len(prompts)):
        row_logits = logits[i]
        row_probs = probs[i]
        s_val = float(torch.logsumexp(row_logits[yes_ids], dim=-1) - torch.logsumexp(row_logits[no_ids], dim=-1))
        mass_val = float(row_probs[yes_ids].sum() + row_probs[no_ids].sum())
        n_tok = int(inputs['attention_mask'][i].sum().item())
        batch_results.append((s_val, mass_val, n_tok))
    return batch_results


In [ ]:
# Step 6: Evaluation Protocol Runner Across All Tasks
import tqdm

BATCH_SIZE = 16
OUTPUT_CHECKPOINT = Path('/kaggle/working/gemma4_e4b_checkpoint.json')
FINAL_RESULTS_JSON = Path('/kaggle/working/gemma4_e4b_reranking_results.json')
FINAL_SCORES_PARQUET = Path('/kaggle/working/gemma4_e4b_scores.parquet')

# Load existing checkpoint if resuming
scored_tasks = {}
if OUTPUT_CHECKPOINT.exists():
    try:
        with open(OUTPUT_CHECKPOINT, 'r', encoding='utf-8') as f:
            scored_tasks = json.load(f)
        print(f'[+] Resuming from checkpoint: {len(scored_tasks)} tasks already evaluated.')
    except Exception:
        scored_tasks = {}

parquet_rows = []
start_time = time.time()

for t_idx, task in enumerate(tasks):
    iid = task['instance_id']
    if iid in scored_tasks:
        continue

    q_text = truncate_text((task.get('problem_statement') or '') + ' ' + (task.get('hints_text') or ''))
    candidates = task.get('candidates', [])
    gold_ids = set(task.get('gold_symbol_ids', []))
    gold_names = set(task.get('gold_symbol_names', []))

    if not candidates and 'arms' in task:
        # Fallback for packaged batch format
        candidates = []
        for arm_name, arm_pack in task['arms'].items():
            for c in arm_pack.get('candidates', []):
                if c not in candidates:
                    candidates.append(c)

    if not candidates:
        continue

    # Build prompts for all candidates in task
    task_prompts = []
    cand_meta = []
    for c_idx, cand in enumerate(candidates):
        bc = format_breadcrumb(cand)
        prompt = PROMPT_TEMPLATE.format(problem_statement=q_text, candidate_breadcrumb=bc)
        chat_p = tokenizer.apply_chat_template(
            [{'role': 'user', 'content': prompt}],
            tokenize=False,
            add_generation_prompt=True,
        )
        task_prompts.append(chat_p)
        c_id = cand.get('candidate_id', cand.get('node_idx', c_idx))
        cand_meta.append((c_id, cand))

    # Execute batched inference
    task_scores = []
    for b_start in range(0, len(task_prompts), BATCH_SIZE):
        b_prompts = task_prompts[b_start : b_start + BATCH_SIZE]
        b_scored = score_prompt_batch(b_prompts)
        task_scores.extend(b_scored)

    # Compile task record
    ranked_candidates = []
    for (c_id, cand), (s_val, mass_val, n_tok) in zip(cand_meta, task_scores):
        sym_id = cand.get('symbol_id', c_id)
        ident = cand.get('identifier') or cand.get('symbol_name') or ''
        is_gold = (sym_id in gold_ids) or (ident in gold_names) or cand.get('is_gold', False)
        ranked_candidates.append({
            'candidate_id': c_id,
            'symbol_id': sym_id,
            'identifier': ident,
            's': s_val,
            'format_mass': mass_val,
            'n_prompt_tokens': n_tok,
            'is_gold': is_gold,
        })
        parquet_rows.append({
            'instance_id': iid,
            'candidate_id': c_id,
            's': s_val,
            'format_mass': mass_val,
            'n_prompt_tokens': n_tok,
            'is_gold': int(is_gold),
        })

    # Sort descending by log-odds score
    ranked_candidates.sort(key=lambda x: x['s'], reverse=True)

    hit1 = any(c['is_gold'] for c in ranked_candidates[:1])
    hit3 = any(c['is_gold'] for c in ranked_candidates[:3])
    hit5 = any(c['is_gold'] for c in ranked_candidates[:5])
    hit10 = any(c['is_gold'] for c in ranked_candidates[:10])

    gold_ranks = [r + 1 for r, c in enumerate(ranked_candidates) if c['is_gold']]
    rr = 1.0 / gold_ranks[0] if gold_ranks else 0.0

    scored_tasks[iid] = {
        'hit_1': hit1,
        'hit_3': hit3,
        'hit_5': hit5,
        'hit_10': hit10,
        'mrr': rr,
        'total_candidates': len(ranked_candidates),
        'gold_ranks': gold_ranks,
    }

    # Periodic checkpoint save every 10 tasks
    if (t_idx + 1) % 10 == 0 or (t_idx + 1) == len(tasks):
        with open(OUTPUT_CHECKPOINT, 'w', encoding='utf-8') as f:
            json.dump(scored_tasks, f)
        elapsed = time.time() - start_time
        rate = (t_idx + 1) / max(elapsed, 1e-5)
        print(f'[{t_idx+1}/{len(tasks)}] Tasks Processed | Rate: {rate:.2f} tasks/s | Elapsed: {elapsed/60:.1f}m')

print(f'[+] Completed pointwise scoring across {len(scored_tasks)} tasks.')


In [ ]:
# Step 7: Statistical Analysis & Capacity Floor Breakdown

n_tasks = len(scored_tasks)
if n_tasks > 0:
    acc1 = np.mean([t['hit_1'] for t in scored_tasks.values()]) * 100
    acc3 = np.mean([t['hit_3'] for t in scored_tasks.values()]) * 100
    acc5 = np.mean([t['hit_5'] for t in scored_tasks.values()]) * 100
    acc10 = np.mean([t['hit_10'] for t in scored_tasks.values()]) * 100
    mrr = np.mean([t['mrr'] for t in scored_tasks.values()])

    summary_report = {
        'model': TARGET_MODEL_ID,
        'n_tasks': n_tasks,
        'metrics': {
            'acc_at_1_pct': round(acc1, 2),
            'acc_at_3_pct': round(acc3, 2),
            'acc_at_5_pct': round(acc5, 2),
            'acc_at_10_pct': round(acc10, 2),
            'mrr': round(float(mrr), 4),
        },
        'baselines_comparison': {
            'gemma_4_e2b_it': {'acc_at_1': 24.7, 'acc_at_5': 47.1, 'mrr': 0.3409},
            'unscored_bm25':  {'acc_at_1': 25.3, 'acc_at_5': 44.8, 'mrr': 0.3439},
        },
        'delta_vs_e2b': {
            'delta_acc1_pp': round(acc1 - 24.7, 2),
            'delta_acc5_pp': round(acc5 - 47.1, 2),
            'delta_mrr': round(float(mrr) - 0.3409, 4),
        }
    }

    # Save final artifacts
    with open(FINAL_RESULTS_JSON, 'w', encoding='utf-8') as f:
        json.dump({'summary': summary_report, 'tasks': scored_tasks}, f, indent=2)
    print(f'[+] Saved final JSON results to: {FINAL_RESULTS_JSON}')

    if parquet_rows:
        df_scores = pd.DataFrame(parquet_rows)
        df_scores.to_parquet(FINAL_SCORES_PARQUET, index=False)
        print(f'[+] Saved final parquet scores to: {FINAL_SCORES_PARQUET} ({len(df_scores)} rows)')

    print('\n' + '='*65)
    print('  GEMMA 4 E4B POSITIVE CONTROL: BENCHMARK SUMMARY TABLE')
    print('='*65)
    print(f'Tasks Evaluated : {n_tasks}')
    print(f'Top-1 Accuracy  : {acc1:.2f}% (E2B: 24.70% | BM25: 25.30%) -> Delta: {acc1 - 24.7:+.2f} pp')
    print(f'Top-3 Accuracy  : {acc3:.2f}%')
    print(f'Top-5 Accuracy  : {acc5:.2f}% (E2B: 47.10% | BM25: 44.80%) -> Delta: {acc5 - 47.1:+.2f} pp')
    print(f'Top-10 Accuracy : {acc10:.2f}%')
    print(f'MRR             : {mrr:.4f} (E2B: 0.3409 | BM25: 0.3439) -> Delta: {mrr - 0.3409:+.4f}')
    print('='*65)
